# Vauban — Aircraft Detection & Classification Training

**Stage 1**: YOLOv8 fine-tuned on MAR20 satellite imagery → `vauban_detect_v1.pt`

**Stage 2**: EfficientNet-B0 classifier on 35 aircraft types → `vauban_classifier_v1.pt`

## Instructions
1. Upload `mar20_yolo.zip` and `aircraft_classifier.zip` to Colab (Files panel on the left)
2. Run all cells
3. Download both `.pt` files and the metrics PNGs from `metrics/`

In [ ]:
!pip install -q ultralytics torch torchvision scikit-learn matplotlib seaborn

## Unzip Datasets
Upload the two zip files first, then run this cell.

In [ ]:
import os, zipfile
from pathlib import Path

os.makedirs('datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('metrics', exist_ok=True)

for name in ['mar20_yolo', 'aircraft_classifier']:
    zip_path = f'{name}.zip'
    if not os.path.exists(zip_path):
        zip_path = f'/content/{name}.zip'
    if not os.path.exists(zip_path):
        print(f'ERROR: {name}.zip not found! Upload it to Colab first.')
        continue
    if os.path.exists(f'datasets/{name}'):
        print(f'{name} already extracted, skipping')
        continue
    print(f'Extracting {zip_path}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('datasets')
    print(f'  Done: datasets/{name}/')

!ls datasets/mar20_yolo/images/ 2>/dev/null && echo '---' && ls datasets/aircraft_classifier/ | head -5

## Fix data.yaml path

In [ ]:
import yaml

yaml_path = 'datasets/mar20_yolo/data.yaml'
with open(yaml_path) as f:
    data = yaml.safe_load(f)

data['path'] = os.path.abspath('datasets/mar20_yolo')

with open(yaml_path, 'w') as f:
    yaml.dump(data, f)

print('Updated data.yaml path to:', data['path'])
print('Classes:', data['names'])

## Stage 1: Train YOLOv8 Detector

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import shutil

model = YOLO('yolov8n.pt')

results = model.train(
    data='datasets/mar20_yolo/data.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    project='runs/detection',
    name='vauban_detect_v1',
    freeze=10,
    lr0=0.001,
    lrf=0.01,
    degrees=180.0,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.3,
)

best = Path('runs/detection/vauban_detect_v1/weights/best.pt')
if best.exists():
    shutil.copy(best, 'models/vauban_detect_v1.pt')
    print('Stage 1 model saved to models/vauban_detect_v1.pt')

## Stage 1: Metrics & Evaluation

In [ ]:
import shutil
from IPython.display import display, Image as IPImage

# YOLO saves its own plots during training
yolo_run = Path('runs/detection/vauban_detect_v1')

# Copy YOLO's built-in metric plots
for name in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png',
             'F1_curve.png', 'P_curve.png', 'R_curve.png', 'PR_curve.png',
             'labels.jpg', 'val_batch0_pred.jpg', 'val_batch1_pred.jpg']:
    src = yolo_run / name
    if src.exists():
        shutil.copy(src, f'metrics/stage1_{name}')

# Display key metrics
print('=== STAGE 1: YOLOv8 Detection Metrics ===')
print()

for title, fname in [
    ('Training Curves (loss, mAP, precision, recall)', 'results.png'),
    ('F1 Score vs Confidence', 'F1_curve.png'),
    ('Precision-Recall Curve', 'PR_curve.png'),
    ('Confusion Matrix', 'confusion_matrix_normalized.png'),
    ('Validation Predictions', 'val_batch0_pred.jpg'),
]:
    p = yolo_run / fname
    if p.exists():
        print(f'\n--- {title} ---')
        display(IPImage(filename=str(p), width=800))
    else:
        print(f'{fname} not found')

In [ ]:
# Run validation and print mAP summary
from ultralytics import YOLO

val_model = YOLO('models/vauban_detect_v1.pt')
val_results = val_model.val(data='datasets/mar20_yolo/data.yaml', split='test')

print('\n=== Stage 1 Test Set Results ===')
print(f'mAP@0.5:      {val_results.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val_results.box.map:.4f}')
print(f'Precision:     {val_results.box.mp:.4f}')
print(f'Recall:        {val_results.box.mr:.4f}')
print()
print('Per-class AP@0.5:')
class_names = val_model.names
for i, ap in enumerate(val_results.box.ap50):
    print(f'  {class_names[i]:<15} {ap:.4f}')

## Stage 2: Train EfficientNet Classifier

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder('datasets/aircraft_classifier/train', transform=train_tf)
val_ds = datasets.ImageFolder('datasets/aircraft_classifier/val', transform=val_tf)
test_ds = datasets.ImageFolder('datasets/aircraft_classifier/test', transform=val_tf)
num_classes = len(train_ds.classes)
print(f'Classes ({num_classes}): {train_ds.classes}')
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
import matplotlib.pyplot as plt

cls_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
cls_model.classifier[1] = nn.Linear(cls_model.classifier[1].in_features, num_classes)
cls_model = cls_model.to(device)

for param in cls_model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cls_model.classifier.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

EPOCHS = 50
best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(EPOCHS):
    if epoch == 10:
        print('Unfreezing backbone')
        for param in cls_model.features.parameters():
            param.requires_grad = True
        optimizer = torch.optim.Adam(cls_model.parameters(), lr=0.0001)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    cls_model.train()
    t_loss, t_correct, t_total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = cls_model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        t_loss += loss.item() * imgs.size(0)
        t_correct += (out.argmax(1) == labels).sum().item()
        t_total += imgs.size(0)

    cls_model.eval()
    v_loss, v_correct, v_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = cls_model(imgs)
            loss = criterion(out, labels)
            v_loss += loss.item() * imgs.size(0)
            v_correct += (out.argmax(1) == labels).sum().item()
            v_total += imgs.size(0)

    t_acc = t_correct / t_total
    v_acc = v_correct / v_total
    history['train_loss'].append(t_loss / t_total)
    history['val_loss'].append(v_loss / v_total)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)
    scheduler.step(v_loss / v_total)

    print(f'Epoch {epoch+1}/{EPOCHS} | Train: {t_acc:.4f} | Val: {v_acc:.4f}')

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save({
            'model_state_dict': cls_model.state_dict(),
            'classes': train_ds.classes,
            'num_classes': num_classes,
            'best_val_acc': best_val_acc,
            'epoch': epoch,
        }, 'models/vauban_classifier_v1.pt')
        print(f'  -> Saved (val_acc={v_acc:.4f})')

print(f'\nDone. Best val accuracy: {best_val_acc:.4f}')

## Stage 2: Metrics & Evaluation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', color='#e74c3c')
axes[0].plot(history['val_loss'], label='Val Loss', color='#3498db')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history['train_acc'], label='Train Acc', color='#e74c3c')
axes[1].plot(history['val_acc'], label='Val Acc', color='#3498db')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('metrics/stage2_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Load best model
checkpoint = torch.load('models/vauban_classifier_v1.pt', map_location=device)
cls_model.load_state_dict(checkpoint['model_state_dict'])
cls_model.eval()
class_names = checkpoint['classes']

# Run on test set
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out = cls_model(imgs)
        probs = torch.softmax(out, dim=1)
        preds = out.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Overall metrics
test_acc = (all_preds == all_labels).mean()
f1_macro = f1_score(all_labels, all_preds, average='macro')
f1_weighted = f1_score(all_labels, all_preds, average='weighted')

try:
    auc_macro = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    auc_weighted = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='weighted')
except:
    auc_macro = auc_weighted = None

print('=== STAGE 2: Classifier Test Set Results ===')
print(f'Test Accuracy:    {test_acc:.4f}')
print(f'F1 (macro):       {f1_macro:.4f}')
print(f'F1 (weighted):    {f1_weighted:.4f}')
if auc_macro:
    print(f'AUC-ROC (macro):  {auc_macro:.4f}')
    print(f'AUC-ROC (weighted): {auc_weighted:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=class_names, digits=3))

In [ ]:
# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=ax, vmin=0, vmax=1, square=True)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Normalized Confusion Matrix — Aircraft Classifier', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('metrics/stage2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-class F1 bar chart
from sklearn.metrics import f1_score as f1_per_class

f1_scores = f1_per_class(all_labels, all_preds, average=None)

# Sort by F1 score
sorted_idx = np.argsort(f1_scores)
sorted_names = [class_names[i] for i in sorted_idx]
sorted_f1 = f1_scores[sorted_idx]

# Color Russian types differently
russian = {'Su24','Su25','Su34','Su47','Su57','Mig29','Mig31','Tu95','Tu160','Tu22M',
           'Il76','Ka52','Ka27','Mi8','Mi24','Mi26','Mi28','An124','An22'}
colors = ['#e74c3c' if n in russian else '#3498db' for n in sorted_names]

fig, ax = plt.subplots(figsize=(12, 10))
bars = ax.barh(range(len(sorted_names)), sorted_f1, color=colors)
ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=10)
ax.set_xlabel('F1 Score')
ax.set_title('Per-Class F1 Score (Red = Russian, Blue = NATO)')
ax.set_xlim(0, 1)
ax.grid(True, axis='x', alpha=0.3)

for bar, f1 in zip(bars, sorted_f1):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{f1:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('metrics/stage2_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top confusions — which aircraft types get confused most
np.fill_diagonal(cm, 0)
top_k = 15
flat_idx = np.argsort(cm.ravel())[-top_k:][::-1]
rows, cols = np.unravel_index(flat_idx, cm.shape)

print(f'Top {top_k} confusions:')
print(f'{"True":<15} {"Predicted":<15} {"Count":>6}')
print('-' * 38)
for r, c in zip(rows, cols):
    if cm[r, c] > 0:
        print(f'{class_names[r]:<15} {class_names[c]:<15} {cm[r,c]:>6}')

## Download Models & Metrics
Place `.pt` files in `Vauban/ml/models/` and metric PNGs wherever you want.

In [ ]:
import os

print('=== Models ===')
for f in os.listdir('models'):
    if f.endswith('.pt'):
        size_mb = os.path.getsize(f'models/{f}') / 1024 / 1024
        print(f'  models/{f} — {size_mb:.1f} MB')

print('\n=== Metrics ===')
for f in sorted(os.listdir('metrics')):
    if f.endswith('.png'):
        print(f'  metrics/{f}')

try:
    from google.colab import files
    files.download('models/vauban_detect_v1.pt')
    files.download('models/vauban_classifier_v1.pt')
    for f in os.listdir('metrics'):
        if f.endswith('.png'):
            files.download(f'metrics/{f}')
except:
    print('\nNot on Colab — grab files manually')